# LIDA + Story Categories V2 Final Candidate Merge

This notebook prepares the first-stage final candidate dataset from:

- `data/processed/library_data/story_categories_v2.json` as the cleaned StoryWeaver V2 base
- `data/raw/lida_stories/lida_stories_en.json` as the cleaned LIDA source

This notebook normalizes and cleans the LIDA story dataset to match the StoryWeaver V2 schema and story category structure. The cleaned LIDA stories are then merged into the V2 dataset after duplicate and short-text filtering. The final merged dataset is exported as `outputs/results/iteration3_library/story_categories_v3.json` for Epic 5.

## 1. Setup Paths and Constants

In [31]:
import json
import re
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "library_data" / "story_categories_v2.json").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/processed/library_data/story_categories_v2.json")

# Resolve the project root whether this notebook is run from repo root or notebooks/.
cwd = Path.cwd()
ROOT = find_project_root(cwd)

V2_PATH = ROOT / "data/processed/library_data/story_categories_v2.json"
LIDA_PATH = ROOT / "data/raw/lida_stories/lida_stories_en.json"

FINAL_CANDIDATE_PATH = ROOT / "data/processed/library_data/story_categories_final_candidate.json"
V3_OUTPUT_PATH = ROOT / "outputs/results/iteration3_library/story_categories_v3.json"
TEXT_SIMILARITY_THRESHOLD = 0.95

# V2 fields are the base schema. LIDA license fields are kept so attribution is not lost.
FINAL_COLUMNS = [
    "unified_id",
    "source_dataset",
    "source_id",
    "title",
    "text",
    "word_count",
    "reading_level",
    "reading_level_key",
    "original_category",
    "category_v1",
    "source",
    "source_url",
    "author",
    "illustrator",
    "license_name",
    "license_url",
    "attribution",
]

print("Project root:", ROOT)
print("V2 exists:", V2_PATH.exists(), V2_PATH.relative_to(ROOT))
print("LIDA exists:", LIDA_PATH.exists(), LIDA_PATH.relative_to(ROOT))


Project root: <project_root>
V2 exists: True data/processed/library_data/story_categories_v2.json
LIDA exists: True data/raw/lida_stories/lida_stories_en.json


## 2. Load Inputs

In [32]:
def load_json_records(path):
    """Load a JSON file that should contain a list of records."""
    with path.open(encoding="utf-8") as f:
        records = json.load(f)
    if not isinstance(records, list):
        raise TypeError(f"Expected a list in {path}, got {type(records).__name__}")
    return records


raw_v2_records = load_json_records(V2_PATH)
raw_lida_records = load_json_records(LIDA_PATH)

print("V2 input records:", len(raw_v2_records))
print("LIDA input records:", len(raw_lida_records))
print("V2 fields:", list(raw_v2_records[0].keys()))
print("LIDA fields:", list(raw_lida_records[0].keys()))

assert len(raw_v2_records) == 6942
assert len(raw_lida_records) == 30

V2 input records: 6942
LIDA input records: 30
V2 fields: ['unified_id', 'source_dataset', 'source_id', 'title', 'text', 'word_count', 'reading_level', 'reading_level_key', 'original_category', 'category_v1', 'source', 'source_url', 'author', 'illustrator']
LIDA fields: ['story_id', 'title', 'level', 'text', 'word_count', 'source_url', 'author', 'illustrator', 'reader', 'language', 'license_name', 'license_url', 'attribution', 'modified', 'cleaning_note', 'project_use', 'retrieved_at', 'html_path', 'text_path']


## 3. Normalize LIDA to the V2 Schema

LIDA is reshaped into the same record structure as V2. At this stage, LIDA `category_v1` is intentionally left as `None`.

In [33]:
def normalize_lida_level_key(level):
    """Map LIDA Level 1/2/3/4/5 into the three V2 level buckets."""
    match = re.search(r"\d+", str(level or ""))
    if not match:
        raise ValueError(f"Cannot parse LIDA reading level: {level!r}")

    level_number = int(match.group())
    if level_number <= 1:
        return "level_1"
    if level_number == 2:
        return "level_2"
    return "level_3"


def align_record(record):
    """Return a record with exactly FINAL_COLUMNS, filling missing fields with None."""
    return {column: record.get(column) for column in FINAL_COLUMNS}


def normalize_v2_record(record):
    """Keep V2 values and add LIDA-only attribution columns as None."""
    return align_record(record)


def normalize_lida_record(record):
    """Convert one LIDA source record to the final candidate schema."""
    story_id = str(record.get("story_id", "")).strip()
    if not story_id:
        raise ValueError(f"LIDA record has no story_id: {record!r}")

    return align_record(
        {
            "unified_id": f"lida_{story_id}",
            "source_dataset": "lida_stories",
            "source_id": story_id,
            "title": record.get("title"),
            "text": record.get("text"),
            "word_count": record.get("word_count"),
            "reading_level": record.get("level"),
            "reading_level_key": normalize_lida_level_key(record.get("level")),
            "original_category": None,
            "category_v1": None,
            "source": "LIDA Stories",
            "source_url": record.get("source_url"),
            "author": record.get("author"),
            "illustrator": record.get("illustrator"),
            "license_name": record.get("license_name"),
            "license_url": record.get("license_url"),
            "attribution": record.get("attribution"),
        }
    )


v2_records = [normalize_v2_record(record) for record in raw_v2_records]
lida_records = [normalize_lida_record(record) for record in raw_lida_records]

print("Normalized V2 records:", len(v2_records))
print("Normalized LIDA records:", len(lida_records))
print("LIDA level mapping:")
display(pd.DataFrame(lida_records).groupby(["reading_level", "reading_level_key"]).size().reset_index(name="count"))
display(pd.DataFrame(lida_records).head())

Normalized V2 records: 6942
Normalized LIDA records: 30
LIDA level mapping:


,reading_level,reading_level_key,count
0,Level 1,level_1,10
1,Level 2,level_2,8
2,Level 3,level_3,4
3,Level 4,level_3,4
4,Level 5,level_3,4


,unified_id,source_dataset,source_id,title,text,word_count,reading_level,reading_level_key,original_category,category_v1,source,source_url,author,illustrator,license_name,license_url,attribution
0,lida_0023,lida_stories,0023,The weather,It is sunny.\n\nIt is cloudy.\n\nIt is raining.\n\nIt is windy.\n\nIt is snowing.\n\nIt is cold.\n\nIt is hot.,21,Level 1,level_1,None,None,LIDA Stories,https://lidastories.net/uk/stories/en/0023/,Espen Stranger-Johannessen,"Tuva Akselsen Sandersen, Josefine Madsen",Creative Commons Attribution 4.0 International License,https://creativecommons.org/licenses/by/4.0/,"The weather by Espen Stranger-Johannessen, from LIDA Stories (https://lidastories.net/uk/stories/en/0023/), licensed under Creative Commons Attribution 4.0 International License (https://creativecommons.org/licenses/by/4.0/)."
1,lida_0002,lida_stories,0002,Feelings,I am scared.\n\nOuch!\n\nAre you okay?\n\nMy stomach hurts.\n\nI have a headache.\n\nI’m hungry.\n\nI’m tired.\n\nI’m sleepy.\n\nI’m thirsty.\n\nThat’s a shame.,30,Level 1,level_1,None,None,LIDA Stories,https://lidastories.net/uk/stories/en/0002/,Espen Stranger-Johannessen,Aakanee,Creative Commons Attribution 4.0 International License,https://creativecommons.org/licenses/by/4.0/,"Feelings by Espen Stranger-Johannessen, from LIDA Stories (https://lidastories.net/uk/stories/en/0002/), licensed under Creative Commons Attribution 4.0 International License (https://creativecommons.org/licenses/by/4.0/)."
2,lida_0003,lida_stories,0003,Asking and answering,"I would like that one, please.\n\nDo you have…?\n\nIs it okay if I sit here?\n\nThat will be fine.\n\nWould you like some of this?\n\nYes, please!\n\nCould I see that one, please?\n\nHere you are.",37,Level 1,level_1,None,None,LIDA Stories,https://lidastories.net/uk/stories/en/0003/,Espen Stranger-Johannessen,Aakanee,Creative Commons Attribution 4.0 International License,https://creativecommons.org/licenses/by/4.0/,"Asking and answering by Espen Stranger-Johannessen, from LIDA Stories (https://lidastories.net/uk/stories/en/0003/), licensed under Creative Commons Attribution 4.0 International License (https://creativecommons.org/licenses/by/4.0/)."
3,lida_0022,lida_stories,0022,What are they doing?,I am talking.\n\nYou are walking.\n\nHe is writing.\n\nShe is sleeping.\n\nIt is eating.\n\nWe are reading.\n\nYou are listening.\n\nThey are driving.\n\nEveryone is doing something!,28,Level 1,level_1,None,None,LIDA Stories,https://lidastories.net/uk/stories/en/0022/,Espen Stranger-Johannessen,Frida Sofie Schønsberg,Creative Commons Attribution 4.0 International License,https://creativecommons.org/licenses/by/4.0/,"What are they doing? by Espen Stranger-Johannessen, from LIDA Stories (https://lidastories.net/uk/stories/en/0022/), licensed under Creative Commons Attribution 4.0 International License (https://creativecommons.org/licenses/by/4.0/)."
4,lida_0010,lida_stories,0010,Clean and ready,He washes his hands.\n\nHe washes his face.\n\nHe brushes his teeth.\n\nHe shaves.\n\nHe showers.\n\nHe washes his hair.\n\nHe dries himself.\n\nHe combs his hair.\n\nHe cuts his fingernails.,31,Level 1,level_1,None,None,LIDA Stories,https://lidastories.net/uk/stories/en/0010/,Espen Stranger-Johannessen,Aakanee,Creative Commons Attribution 4.0 International License,https://creativecommons.org/licenses/by/4.0/,"Clean and ready by Espen Stranger-Johannessen, from LIDA Stories (https://lidastories.net/uk/stories/en/0010/), licensed under Creative Commons Attribution 4.0 International License (https://creativecommons.org/licenses/by/4.0/)."


## 4. Cross-Source Duplicate Detection

Only LIDA-vs-V2 duplicates are checked. This follows the V2 cleaning notebook's duplicate logic as closely as possible: title candidates first, then text comparison.

Duplicate rules:

- first find exact or highly similar title pairs across sources
- then compare text only inside those title candidate pairs
- normalized text is exactly the same: delete LIDA and keep V2
- normalized text similarity is `>= 0.95`: write to review only, do not delete automatically

This means high text similarity behaves like V2: it is evidence for review, while automatic deletion is limited to exact same-text duplicates.

In [34]:
def normalize_text_for_duplicate_check(text):
    """Match V2's text comparison: lowercase, replace newlines, and trim."""
    return str(text or "").lower().replace("\n", " ").strip()


def normalize_title_for_duplicate_check(title):
    """Match V2's title normalization for duplicate/similar-title candidates."""
    return re.sub(r"[^a-z0-9]+", " ", str(title or "").lower()).strip()


def make_title_key(normalized_title):
    """Use the same small-bucket title key shape used in the V2 notebook."""
    words = normalized_title.split()
    first_word = words[0] if words else ""
    prefix = normalized_title[:8]
    length_bucket = len(normalized_title) // 5
    return (first_word, prefix, length_bucket)


def safe_word_count(record):
    """Use stored word_count when possible; otherwise recalculate from text."""
    try:
        return int(record.get("word_count") or 0)
    except (TypeError, ValueError):
        return len(str(record.get("text") or "").split())


def build_title_candidate_pairs(v2_records, lida_records):
    """Find LIDA-vs-V2 pairs whose titles are exact or highly similar matches."""
    v2_by_normalized_title = defaultdict(list)
    v2_by_title_key = defaultdict(list)

    for record in v2_records:
        normalized_title = normalize_title_for_duplicate_check(record.get("title"))
        if not normalized_title:
            continue
        v2_by_normalized_title[normalized_title].append(record)
        v2_by_title_key[make_title_key(normalized_title)].append((normalized_title, record))

    candidate_pairs = []
    seen_pairs = set()
    title_similarity_threshold = 0.9

    def add_pair(lida_record, v2_record, match_type, title_similarity):
        pair_key = (lida_record["unified_id"], v2_record["unified_id"])
        if pair_key in seen_pairs:
            return
        seen_pairs.add(pair_key)
        candidate_pairs.append((lida_record, v2_record, match_type, title_similarity))

    for lida_record in lida_records:
        lida_title = normalize_title_for_duplicate_check(lida_record.get("title"))
        if not lida_title:
            continue

        # 1. Exact normalized title matches.
        for v2_record in v2_by_normalized_title.get(lida_title, []):
            add_pair(lida_record, v2_record, "exact_title_match", 1.0)

        # 2. Similar title matches inside the same small bucket.
        for v2_title, v2_record in v2_by_title_key.get(make_title_key(lida_title), []):
            if lida_title == v2_title or abs(len(lida_title) - len(v2_title)) > 8:
                continue
            title_similarity = SequenceMatcher(None, lida_title, v2_title).ratio()
            if title_similarity >= title_similarity_threshold:
                add_pair(lida_record, v2_record, "similar_title_match", title_similarity)

    return candidate_pairs


def find_cross_source_duplicates(v2_records, lida_records):
    """Compare text only for exact/similar-title candidate pairs."""
    duplicate_rows = []
    deleted_lida_ids = set()
    seen_pairs = set()

    candidate_pairs = build_title_candidate_pairs(v2_records, lida_records)

    def add_review_row(lida_record, v2_record, title_match_type, title_similarity, text_similarity, reason, action):
        pair_key = (lida_record["unified_id"], v2_record["unified_id"])
        if pair_key in seen_pairs:
            return
        seen_pairs.add(pair_key)

        duplicate_rows.append(
            {
                "lida_unified_id": lida_record["unified_id"],
                "lida_source_id": lida_record["source_id"],
                "lida_title": lida_record["title"],
                "lida_word_count": safe_word_count(lida_record),
                "v2_unified_id": v2_record["unified_id"],
                "v2_source_id": v2_record["source_id"],
                "v2_title": v2_record["title"],
                "v2_word_count": safe_word_count(v2_record),
                "title_match_type": title_match_type,
                "title_similarity": round(title_similarity, 6),
                "text_similarity": round(text_similarity, 6),
                "duplicate_reason": reason,
                "action": action,
            }
        )
        if action == "delete_lida_keep_v2":
            deleted_lida_ids.add(lida_record["unified_id"])

    for lida_record, v2_record, title_match_type, title_similarity in candidate_pairs:
        lida_text = normalize_text_for_duplicate_check(lida_record.get("text"))
        v2_text = normalize_text_for_duplicate_check(v2_record.get("text"))
        if not lida_text or not v2_text:
            continue

        text_similarity = SequenceMatcher(None, lida_text, v2_text).ratio()
        if lida_text == v2_text:
            add_review_row(
                lida_record,
                v2_record,
                title_match_type,
                title_similarity,
                1.0,
                "same_text",
                "delete_lida_keep_v2",
            )
        elif text_similarity >= TEXT_SIMILARITY_THRESHOLD:
            add_review_row(
                lida_record,
                v2_record,
                title_match_type,
                title_similarity,
                text_similarity,
                f"text_similarity_ge_{TEXT_SIMILARITY_THRESHOLD}",
                "review_only_keep_both",
            )

    duplicate_rows = sorted(
        duplicate_rows,
        key=lambda row: (row["lida_source_id"], -float(row["text_similarity"]), row["v2_source_id"]),
    )
    return duplicate_rows, deleted_lida_ids


duplicate_rows, deleted_lida_ids = find_cross_source_duplicates(v2_records, lida_records)

title_candidate_pairs = build_title_candidate_pairs(v2_records, lida_records)
print("Title candidate pairs checked:", len(title_candidate_pairs))
print("Duplicate/high-similarity review rows:", len(duplicate_rows))
print("Unique LIDA records marked for deletion:", len(deleted_lida_ids))
display(pd.DataFrame(duplicate_rows).head(20))

Title candidate pairs checked: 20
Duplicate/high-similarity review rows: 0
Unique LIDA records marked for deletion: 0


""


## 5. Build Final Candidate and Display Short-Text Review

The candidate keeps V2 first, then appends non-duplicate LIDA records. LIDA records under 25 words are deleted from the candidate. The short-text table is displayed here for transparency and is not saved as a separate CSV file.

In [35]:
deduped_lida_records = [
    record for record in lida_records if record["unified_id"] not in deleted_lida_ids
]

short_text_delete_threshold = 25
short_text_rows = []
deleted_short_lida_ids = set()

for record in deduped_lida_records:
    words = safe_word_count(record)
    if words < short_text_delete_threshold:
        deleted_short_lida_ids.add(record["unified_id"])
        short_text_rows.append(
            {
                "unified_id": record["unified_id"],
                "source_id": record["source_id"],
                "title": record["title"],
                "reading_level": record["reading_level"],
                "reading_level_key": record["reading_level_key"],
                "word_count": words,
                "source_url": record["source_url"],
                "action": "delete_lida_under_25_words",
            }
        )

kept_lida_records = [
    record for record in deduped_lida_records if record["unified_id"] not in deleted_short_lida_ids
]
final_candidate_records = v2_records + kept_lida_records

short_text_rows = sorted(short_text_rows, key=lambda row: (row["word_count"], row["source_id"]))

print("Final candidate records:", len(final_candidate_records))
print("LIDA records after duplicate deletion:", len(deduped_lida_records))
print("LIDA records deleted for <25 words:", len(deleted_short_lida_ids))
print("Remaining LIDA records:", len(kept_lida_records))
display(pd.DataFrame(short_text_rows))

Final candidate records: 6971
LIDA records after duplicate deletion: 30
LIDA records deleted for <25 words: 1
Remaining LIDA records: 29


,unified_id,source_id,title,reading_level,reading_level_key,word_count,source_url,action
0,lida_0023,0023,The weather,Level 1,level_1,21,https://lidastories.net/uk/stories/en/0023/,delete_lida_under_25_words


## 6. Validate Candidate Dataset

These checks make sure the candidate file is structurally safe before writing it.

In [36]:
expected_count = len(raw_v2_records) + len(raw_lida_records) - len(deleted_lida_ids) - len(deleted_short_lida_ids)

assert len(raw_v2_records) == 6942
assert len(raw_lida_records) == 30
assert len(final_candidate_records) == expected_count

unified_ids = [record["unified_id"] for record in final_candidate_records]
assert len(unified_ids) == len(set(unified_ids)), "unified_id values must be unique"

field_sets = {tuple(record.keys()) for record in final_candidate_records}
assert field_sets == {tuple(FINAL_COLUMNS)}, "all records must use exactly FINAL_COLUMNS"

level_keys = {record["reading_level_key"] for record in final_candidate_records}
assert level_keys <= {"level_1", "level_2", "level_3"}

for record in final_candidate_records:
    if record["source_dataset"] == "lida_stories":
        assert record["category_v1"] is None
    else:
        assert record["category_v1"] is not None

print("Validation passed")
print("Source counts:", Counter(record["source_dataset"] for record in final_candidate_records))
print("Category counts:", Counter(record["category_v1"] for record in final_candidate_records))
print("Reading level key counts:", Counter(record["reading_level_key"] for record in final_candidate_records))

Validation passed
Source counts: Counter({'storyweaver': 6942, 'lida_stories': 29})
Category counts: Counter({'Daily Life': 3807, 'Animals': 2739, 'Science & Knowledge': 396, None: 29})
Reading level key counts: Counter({'level_2': 2505, 'level_3': 2315, 'level_1': 2151})


## 7. Write Candidate File

This is the only step that writes a file. Duplicate and short-text review tables are displayed above, not saved separately.

In [37]:
def write_json_records(path, records):
    """Write records as UTF-8 pretty JSON."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

write_json_records(FINAL_CANDIDATE_PATH, final_candidate_records)

print("Wrote", FINAL_CANDIDATE_PATH.relative_to(ROOT))

Wrote data/processed/library_data/story_categories_final_candidate.json


## 8. LIDA Category Analysis

This section does not write a final labeled dataset. It uses the existing V2 `category_v1` labels as reference data to produce explainable category evidence for the remaining LIDA records.

Evidence sources:

- TF-IDF + Logistic Regression predictions trained on V2
- top-5 nearest V2 stories in the same TF-IDF space
- model-weighted explanatory terms from each LIDA text

The output is an analysis table for review before any LIDA `category_v1` values are written.

In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import make_pipeline

analysis_df = pd.DataFrame(final_candidate_records).copy()
reference_df = analysis_df[analysis_df["source_dataset"] == "storyweaver"].copy()
target_lida_df = analysis_df[analysis_df["source_dataset"] == "lida_stories"].copy()

def combine_title_text(df):
    return (df["title"].fillna("") + " " + df["text"].fillna("")).str.replace(r"\s+", " ", regex=True).str.strip()

X_reference_text = combine_title_text(reference_df)
y_reference = reference_df["category_v1"]
X_lida_text = combine_title_text(target_lida_df)

print("Reference records:", len(reference_df))
print("Target LIDA records:", len(target_lida_df))
print("Reference category distribution:")
display(y_reference.value_counts().rename_axis("category_v1").reset_index(name="count"))

Reference records: 6942
Target LIDA records: 29
Reference category distribution:


,category_v1,count
0,Daily Life,3807
1,Animals,2739
2,Science & Knowledge,396


### 8.1 Validate the Classifier on V2

Before applying the model to LIDA, cross-validate it on V2 so we know how reliable this signal is. The model is intentionally simple and interpretable: TF-IDF n-grams plus logistic regression.

In [39]:
category_model = make_pipeline(
    TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_features=50000,
    ),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(category_model, X_reference_text, y_reference, cv=cv, scoring="f1_macro")
cv_predictions = cross_val_predict(category_model, X_reference_text, y_reference, cv=cv)

print("5-fold macro F1 scores:", [round(score, 3) for score in cv_scores])
print("Mean macro F1:", round(float(cv_scores.mean()), 3))

print("Classification report:")
print(classification_report(y_reference, cv_predictions, zero_division=0))

labels = sorted(y_reference.unique())
confusion_df = pd.DataFrame(
    confusion_matrix(y_reference, cv_predictions, labels=labels),
    index=[f"actual_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels],
)
display(confusion_df)

5-fold macro F1 scores: [0.682, 0.731, 0.716, 0.698, 0.729]
Mean macro F1: 0.711
Classification report:
                     precision    recall  f1-score   support

            Animals       0.85      0.83      0.84      2739
         Daily Life       0.85      0.87      0.86      3807
Science & Knowledge       0.44      0.42      0.43       396

           accuracy                           0.83      6942
          macro avg       0.72      0.71      0.71      6942
       weighted avg       0.83      0.83      0.83      6942



,pred_Animals,pred_Daily Life,pred_Science & Knowledge
actual_Animals,2282,401,56
actual_Daily Life,352,3301,154
actual_Science & Knowledge,53,175,168


### 8.2 Predict LIDA Categories and Retrieve Similar V2 Stories

The model is trained on all V2 records, then applied to LIDA. In parallel, each LIDA story is compared to V2 stories in the TF-IDF vector space, and the top 5 neighbors are used as a second source of category evidence.

In [40]:
# Fit the interpretable model on all V2 records.
category_model.fit(X_reference_text, y_reference)
vectorizer = category_model.named_steps["tfidfvectorizer"]
classifier = category_model.named_steps["logisticregression"]
classes = classifier.classes_

lida_probabilities = classifier.predict_proba(vectorizer.transform(X_lida_text))

# Fit a shared vector space for nearest-neighbor evidence.
reference_matrix = vectorizer.transform(X_reference_text)
lida_matrix = vectorizer.transform(X_lida_text)
similarity_matrix = cosine_similarity(lida_matrix, reference_matrix)

def top_explanatory_terms_for_row(text, class_name, top_n=8):
    """Return terms with the strongest positive contribution for the chosen class."""
    row_vector = vectorizer.transform([text])
    class_index = list(classes).index(class_name)
    coefficients = classifier.coef_[class_index]
    feature_names = vectorizer.get_feature_names_out()

    contributions = row_vector.multiply(coefficients).tocoo()
    term_scores = [
        (feature_names[col], score)
        for col, score in zip(contributions.col, contributions.data)
        if score > 0
    ]
    term_scores = sorted(term_scores, key=lambda item: item[1], reverse=True)[:top_n]
    return ", ".join(term for term, _ in term_scores)

ambiguous_terms = {"weather", "food", "medicine", "birds", "bird", "family", "work"}
analysis_rows = []

for lida_position, (_, lida_row) in enumerate(target_lida_df.reset_index(drop=True).iterrows()):
    probabilities = lida_probabilities[lida_position]
    ranked_class_indices = probabilities.argsort()[::-1]
    model_category = classes[ranked_class_indices[0]]
    second_category = classes[ranked_class_indices[1]]
    model_confidence = float(probabilities[ranked_class_indices[0]])
    second_confidence = float(probabilities[ranked_class_indices[1]])
    confidence_gap = model_confidence - second_confidence

    top_neighbor_indices = similarity_matrix[lida_position].argsort()[::-1][:5]
    top_neighbors = reference_df.iloc[top_neighbor_indices].copy()
    top_neighbor_scores = similarity_matrix[lida_position][top_neighbor_indices]
    neighbor_category_counts = Counter(top_neighbors["category_v1"])
    nearest_majority_category, nearest_majority_count = neighbor_category_counts.most_common(1)[0]

    top_similar_titles = " | ".join(
        f"{row.category_v1}: {row.title} ({score:.3f})"
        for row, score in zip(top_neighbors.itertuples(index=False), top_neighbor_scores)
    )
    nearest_neighbor_categories = ", ".join(
        f"{category}:{count}" for category, count in neighbor_category_counts.most_common()
    )

    top_terms = top_explanatory_terms_for_row(X_lida_text.iloc[lida_position], model_category)
    text_tokens = set(re.findall(r"[a-z]+", str(X_lida_text.iloc[lida_position]).lower()))
    ambiguous_term_hits = sorted(text_tokens & ambiguous_terms)

    auto_suggest = (
        model_confidence >= 0.60
        and confidence_gap >= 0.20
        and nearest_majority_count >= 3
        and nearest_majority_category == model_category
    )

    analysis_rows.append(
        {
            "source_id": lida_row["source_id"],
            "title": lida_row["title"],
            "word_count": lida_row["word_count"],
            "reading_level": lida_row["reading_level"],
            "model_category": model_category,
            "model_confidence": round(model_confidence, 3),
            "second_category": second_category,
            "second_confidence": round(second_confidence, 3),
            "confidence_gap": round(confidence_gap, 3),
            "nearest_neighbor_categories": nearest_neighbor_categories,
            "nearest_majority_category": nearest_majority_category,
            "nearest_majority_count": nearest_majority_count,
            "top_similar_titles": top_similar_titles,
            "top_explanatory_terms": top_terms,
            "ambiguous_term_hits": ", ".join(ambiguous_term_hits),
            "suggested_category": model_category if auto_suggest else None,
            "needs_review": not auto_suggest,
        }
    )

lida_category_analysis_df = pd.DataFrame(analysis_rows).sort_values(
    ["needs_review", "source_id"],
    ascending=[False, True],
)

print("Suggested category counts:")
display(lida_category_analysis_df["suggested_category"].value_counts(dropna=False).rename_axis("suggested_category").reset_index(name="count"))
print("Needs review distribution:")
display(lida_category_analysis_df["needs_review"].value_counts(dropna=False).rename_axis("needs_review").reset_index(name="count"))
print("Needs review count:", int(lida_category_analysis_df["needs_review"].sum()))
display(lida_category_analysis_df)

Suggested category counts:


,suggested_category,count
0,None,17
1,Daily Life,12


Needs review distribution:


,needs_review,count
0,True,17
1,False,12


Needs review count: 17


,source_id,title,word_count,reading_level,model_category,model_confidence,second_category,second_confidence,confidence_gap,nearest_neighbor_categories,nearest_majority_category,nearest_majority_count,top_similar_titles,top_explanatory_terms,ambiguous_term_hits,suggested_category,needs_review
0,0002,Feelings,30,Level 1,Animals,0.547,Daily Life,0.320,0.226,"Animals:4, Daily Life:1",Animals,4,"Animals: Papa Knows a Lot, but Mamma Knows Everything (0.115) | Animals: The Thirsty Camel (0.103) | Animals: Hungry on the Steps (0.101) | Animals: The Little Sleepy Bear (0.100) | Daily Life: Back to School (0.096)","thirsty, hungry, scared, sleepy, ouch, tired, stomach, hurts",,None,True
1,0003,Asking and answering,37,Level 1,Daily Life,0.436,Animals,0.286,0.150,"Daily Life:3, Animals:2",Daily Life,3,"Animals: No More No! (0.197) | Daily Life: Appu's Exam Fear (0.122) | Animals: Piggy Wants to Fly (0.103) | Daily Life: Chhakuli wants to know (0.096) | Daily Life: Me, Myself & I (0.093)","okay, answering, like",,None,True
7,0004,Meeting people,49,Level 1,Daily Life,0.571,Animals,0.279,0.292,"Daily Life:4, Animals:1",Daily Life,4,Daily Life: sara's grandparents (0.276) | Animals: Animal kingdom (0.206) | Daily Life: Hello (0.205) | Daily Life: The two best friends (0.182) | Daily Life: Future plans (0.178),"sara, ali, sorry, nice, moment, come, phone, meeting",,None,True
8,0005,Helping and understanding,42,Level 1,Science & Knowledge,0.410,Daily Life,0.335,0.075,"Science & Knowledge:3, Daily Life:2",Science & Knowledge,3,Science & Knowledge: I Know My Rights (0.128) | Science & Knowledge: I Know My Rights (0.125) | Daily Life: I wanna help (0.113) | Science & Knowledge: A Helping Hand (0.109) | Daily Life: May I can help you! (0.106),"yes, right, helping, help, understand, way, don, know",,None,True
13,0006,Eating out,90,Level 2,Daily Life,0.574,Animals,0.272,0.302,"Daily Life:3, Animals:2",Daily Life,3,Daily Life: Sharing (0.292) | Animals: PANDA (0.189) | Animals: The Stork and the Spaghetti (0.168) | Daily Life: Grandpa ' s Ramen (0.167) | Daily Life: One Day in Han River (0.164),"friend, noodles, waiter, table, brings, feels, sauce, asks",food,None,True
19,0009,Accident,112,Level 3,Daily Life,0.463,Animals,0.317,0.146,"Daily Life:4, Animals:1",Daily Life,4,Daily Life: Saudade (0.179) | Daily Life: How can you be a Marist (0.154) | Daily Life: Saudate (0.154) | Animals: Aakash and his mother (0.142) | Daily Life: Serious Accident (0.101),"carlos, accident, parents, quickly, worried, rides, comes, come",,None,True
20,0011,Going to the cinema,127,Level 3,Daily Life,0.538,Animals,0.245,0.292,"Daily Life:4, Science & Knowledge:1",Daily Life,4,Science & Knowledge: The World That Mai Built (0.260) | Daily Life: Mai Jingjing (0.239) | Daily Life: Through Nizams' Land (0.208) | Daily Life: Defeat The Pain (0.151) | Daily Life: Lisa And The Bucket (0.150),"watch, say, lisa, best, friend, buys, going, favourite",,None,True
10,0012,Going to work,76,Level 2,Daily Life,0.564,Science & Knowledge,0.249,0.315,"Animals:3, Daily Life:2",Animals,3,Animals: Myra's routine (0.144) | Daily Life: The Rhythm of My Day (0.137) | Animals: Lazy Frog (0.105) | Daily Life: The Most Important Work Of All (0.104) | Animals: The Little Ant (0.093),"hair, going, puts, dressed, shower, gets, picks, time",work,None,True
12,0013,Cleaning,90,Level 2,Daily Life,0.578,Animals,0.266,0.312,"Daily Life:4, Animals:1",Daily Life,4,Daily Life: Our Living Room (0.193) | Daily Life: My Home Needs Cleaning (0.192) | Daily Life: A Cloud Full of Trash (0.175) | Daily Life: Lisa And The Bucket (0.167) | Animals: Earth (0.137),"floor, room, puts, bag, needs, clean, window, nice",,None,True
21,0014,Malik’s story,205,Level 4,Science & Knowledge,0.540,Daily Life,0.361,0.179,Daily Life:5,Daily Life,5,Daily Life: Me and my family (0.152) | Daily Life: My Family (0.139) | Daily Life: Language and Script! (0.133) | Daily Life: A PYRRHIC VICTORY (0.132) | Daily Life: Learning to help (0.100),"ye

### 8.3 Review-Focused Views

These views make the uncertain records easier to inspect without writing labels back to the dataset.

In [41]:
needs_review_df = lida_category_analysis_df[lida_category_analysis_df["needs_review"]].copy()
high_confidence_df = lida_category_analysis_df[~lida_category_analysis_df["needs_review"]].copy()

print("High-confidence auto-suggested records:", len(high_confidence_df))
display(high_confidence_df[[
    "source_id",
    "title",
    "model_category",
    "model_confidence",
    "confidence_gap",
    "nearest_neighbor_categories",
    "suggested_category",
]])

print("Records needing manual category review:", len(needs_review_df))
display(needs_review_df[[
    "source_id",
    "title",
    "word_count",
    "reading_level",
    "model_category",
    "model_confidence",
    "second_category",
    "confidence_gap",
    "nearest_neighbor_categories",
    "ambiguous_term_hits",
    "top_explanatory_terms",
    "top_similar_titles",
]])

High-confidence auto-suggested records: 12


,source_id,title,model_category,model_confidence,confidence_gap,nearest_neighbor_categories,suggested_category
23,0001,Finding a job,Daily Life,0.645,0.441,"Daily Life:3, Science & Knowledge:1, Animals:1",Daily Life
9,0007,Going to bed,Daily Life,0.681,0.466,"Daily Life:3, Animals:2",Daily Life
14,0008,Going to a café,Daily Life,0.620,0.423,Daily Life:5,Daily Life
3,0010,Clean and ready,Daily Life,0.649,0.429,Daily Life:5,Daily Life
4,0019,My family,Daily Life,0.885,0.813,"Daily Life:4, Animals:1",Daily Life
5,0020,I can do many things,Daily Life,0.658,0.458,"Daily Life:3, Animals:2",Daily Life
18,0021,Buying clothes,Daily Life,0.672,0.476,"Daily Life:4, Animals:1",Daily Life
2,0022,What are they doing?,Daily Life,0.615,0.312,"Daily Life:4, Animals:1",Daily Life
11,0024,What sort of music do you like?,Daily Life,0.618,0.413,"Daily Life:3, Animals:2",Daily Life
17,0025,Talking about family,Daily Life,0.812,0.687,Daily Life:5,Daily Life


Records needing manual category review: 17


,source_id,title,word_count,reading_level,model_category,model_confidence,second_category,confidence_gap,nearest_neighbor_categories,ambiguous_term_hits,top_explanatory_terms,top_similar_titles
0,0002,Feelings,30,Level 1,Animals,0.547,Daily Life,0.226,"Animals:4, Daily Life:1",,"thirsty, hungry, scared, sleepy, ouch, tired, stomach, hurts","Animals: Papa Knows a Lot, but Mamma Knows Everything (0.115) | Animals: The Thirsty Camel (0.103) | Animals: Hungry on the Steps (0.101) | Animals: The Little Sleepy Bear (0.100) | Daily Life: Back to School (0.096)"
1,0003,Asking and answering,37,Level 1,Daily Life,0.436,Animals,0.150,"Daily Life:3, Animals:2",,"okay, answering, like","Animals: No More No! (0.197) | Daily Life: Appu's Exam Fear (0.122) | Animals: Piggy Wants to Fly (0.103) | Daily Life: Chhakuli wants to know (0.096) | Daily Life: Me, Myself & I (0.093)"
7,0004,Meeting people,49,Level 1,Daily Life,0.571,Animals,0.292,"Daily Life:4, Animals:1",,"sara, ali, sorry, nice, moment, come, phone, meeting",Daily Life: sara's grandparents (0.276) | Animals: Animal kingdom (0.206) | Daily Life: Hello (0.205) | Daily Life: The two best friends (0.182) | Daily Life: Future plans (0.178)
8,0005,Helping and understanding,42,Level 1,Science & Knowledge,0.410,Daily Life,0.075,"Science & Knowledge:3, Daily Life:2",,"yes, right, helping, help, understand, way, don, know",Science & Knowledge: I Know My Rights (0.128) | Science & Knowledge: I Know My Rights (0.125) | Daily Life: I wanna help (0.113) | Science & Knowledge: A Helping Hand (0.109) | Daily Life: May I can help you! (0.106)
13,0006,Eating out,90,Level 2,Daily Life,0.574,Animals,0.302,"Daily Life:3, Animals:2",food,"friend, noodles, waiter, table, brings, feels, sauce, asks",Daily Life: Sharing (0.292) | Animals: PANDA (0.189) | Animals: The Stork and the Spaghetti (0.168) | Daily Life: Grandpa ' s Ramen (0.167) | Daily Life: One Day in Han River (0.164)
19,0009,Accident,112,Level 3,Daily Life,0.463,Animals,0.146,"Daily Life:4, Animals:1",,"carlos, accident, parents, quickly, worried, rides, comes, come",Daily Life: Saudade (0.179) | Daily Life: How can you be a Marist (0.154) | Daily Life: Saudate (0.154) | Animals: Aakash and his mother (0.142) | Daily Life: Serious Accident (0.101)
20,0011,Going to the cinema,127,Level 3,Daily Life,0.538,Animals,0.292,"Daily Life:4, Science & Knowledge:1",,"watch, say, lisa, best, friend, buys, going, favourite",Science & Knowledge: The World That Mai Built (0.260) | Daily Life: Mai Jingjing (0.239) | Daily Life: Through Nizams' Land (0.208) | Daily Life: Defeat The Pain (0.151) | Daily Life: Lisa And The Bucket (0.150)
10,0012,Going to work,76,Level 2,Daily Life,0.564,Science & Knowledge,0.315,"Animals:3, Daily Life:2",work,"hair, going, puts, dressed, shower, gets, picks, time",Animals: Myra's routine (0.144) | Daily Life: The Rhythm of My Day (0.137) | Animals: Lazy Frog (0.105) | Daily Life: The Most Important Work Of All (0.104) | Animals: The Little Ant (0.093)
12,0013,Cleaning,90,Level 2,Daily Life,0.578,Animals,0.312,"Daily Life:4, Animals:1",,"floor, room, puts, bag, needs, clean, window, nice",Daily Life: Our Living Room (0.193) | Daily Life: My Home Needs Cleaning (0.192) | Daily Life: A Cloud Full of Trash (0.175) | Daily Life: Lisa And The Bucket (0.167) | Animals: Earth (0.137)
21,0014,Malik’s story,205,Level 4,Science & Knowledge,0.540,Daily Life,0.179,Daily Life:5,"family, food","years, people, war, language, religion, different, new, help",Daily Life: Me and my family (0.152) | Daily Life: My Family (0.139) | Daily Life: Language and Script! (0.133) | Daily Life: A PYRRHIC VICTORY (0.132) | Daily Life: Learning to help (0.100)


## 9. Final LIDA Review Decisions and V3 Output

After reviewing the uncertain LIDA records, two records were removed because they are strongly picture-text based and do not fit the text-focused dataset requirements:

- `lida_0002` / `Feelings`
- `lida_0005` / `Helping and understanding`

Two reviewed records were manually assigned to `Daily Life` because their content is centered on everyday events and personal life experience rather than science/knowledge exposition:

- `lida_0009` / `Accident`
- `lida_0014` / `Malik’s story`

For all other remaining LIDA records, the final category is assigned from the model's `model_category` in the analysis table. This keeps the final labeling rule evidence-based while preserving the manual review decisions above.

In [42]:

# Manual review decisions.
review_delete_source_ids = {"0002", "0005"}
manual_category_overrides = {
    "0009": "Daily Life",
    "0014": "Daily Life",
}

model_category_by_source_id = dict(
    zip(lida_category_analysis_df["source_id"], lida_category_analysis_df["model_category"])
)

v3_records = []
final_lida_assignment_rows = []

for record in final_candidate_records:
    if record["source_dataset"] != "lida_stories":
        v3_records.append(dict(record))
        continue

    source_id = record["source_id"]

    if source_id in review_delete_source_ids:
        final_lida_assignment_rows.append(
            {
                "source_id": source_id,
                "title": record["title"],
                "action": "delete",
                "final_category_v1": None,
                "decision_source": "manual_review_picture_text_based",
            }
        )
        continue

    final_category = manual_category_overrides.get(source_id, model_category_by_source_id[source_id])
    decision_source = "manual_review_override" if source_id in manual_category_overrides else "model_category"

    updated_record = dict(record)
    updated_record["category_v1"] = final_category
    v3_records.append(updated_record)

    final_lida_assignment_rows.append(
        {
            "source_id": source_id,
            "title": record["title"],
            "action": "keep",
            "final_category_v1": final_category,
            "model_category": model_category_by_source_id[source_id],
            "decision_source": decision_source,
        }
    )

final_lida_assignments_df = pd.DataFrame(final_lida_assignment_rows).sort_values(["action", "source_id"])

print("Final LIDA assignment decisions:")
display(final_lida_assignments_df)
print("V3 source counts:")
display(pd.Series([record["source_dataset"] for record in v3_records]).value_counts().rename_axis("source_dataset").reset_index(name="count"))
print("V3 category counts:")
display(pd.Series([record["category_v1"] for record in v3_records]).value_counts(dropna=False).rename_axis("category_v1").reset_index(name="count"))

Final LIDA assignment decisions:


,source_id,title,action,final_category_v1,decision_source,model_category
0,0002,Feelings,delete,None,manual_review_picture_text_based,NaN
8,0005,Helping and understanding,delete,None,manual_review_picture_text_based,NaN
23,0001,Finding a job,keep,Daily Life,model_category,Daily Life
1,0003,Asking and answering,keep,Daily Life,model_category,Daily Life
7,0004,Meeting people,keep,Daily Life,model_category,Daily Life
13,0006,Eating out,keep,Daily Life,model_category,Daily Life
9,0007,Going to bed,keep,Daily Life,model_category,Daily Life
14,0008,Going to a café,keep,Daily Life,model_category,Daily Life
19,0009,Accident,keep,Daily Life,manual_review_override,Daily Life
3,0010,Clean and ready,keep,Daily Life,model_category,Daily Life


V3 source counts:


,source_dataset,count
0,storyweaver,6942
1,lida_stories,27


V3 category counts:


,category_v1,count
0,Daily Life,3831
1,Animals,2741
2,Science & Knowledge,397


### 9.1 Validate and Write V3

The V3 output is the first fully categorized final dataset. It keeps V2 unchanged and writes a new processed JSON file.

In [43]:
expected_v3_count = len(final_candidate_records) - len(review_delete_source_ids)
assert len(v3_records) == expected_v3_count
assert len(v3_records) == 6969

v3_ids = [record["unified_id"] for record in v3_records]
assert len(v3_ids) == len(set(v3_ids)), "V3 unified_id values must be unique"

v3_field_sets = {tuple(record.keys()) for record in v3_records}
assert v3_field_sets == {tuple(FINAL_COLUMNS)}, "V3 records must keep the same schema"

assert all(record["category_v1"] is not None for record in v3_records), "V3 must not contain empty category_v1"
assert not any(record["unified_id"] in {"lida_0002", "lida_0005"} for record in v3_records)
assert any(record["unified_id"] == "lida_0009" and record["category_v1"] == "Daily Life" for record in v3_records)
assert any(record["unified_id"] == "lida_0014" and record["category_v1"] == "Daily Life" for record in v3_records)

write_json_records(V3_OUTPUT_PATH, v3_records)

print("Validation passed")
print("Wrote", V3_OUTPUT_PATH.relative_to(ROOT))

Validation passed
Wrote outputs/results/iteration3_library/story_categories_v3.json
